# Просмотр исследований LNOI

Исходные поля рассчитываются из терминала командой `python -m linbo3 --stage comsol --config run_config_comsol.json`.

Исследования запускаются через `python -m studies.fit_effect`, `python -m studies.unclassified`, `python -m studies.spatial_single` или `python -m studies.pulse_series`. В следующей ячейке выбери папку отчёта с помощью `STUDY`. Ноутбук читает готовые файлы и показывает JSI и фазу по компактной JSA.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from linbo3.artifacts import read_json

RESULTS = Path("results")
STUDY = "fit-effect"  # fit-effect-w04-w07, unclassified, spatial-single, pulse-series

report = RESULTS / "studies" / STUDY
if not report.is_dir():
    # После аварийного завершения при замене может остаться предыдущий отчёт.
    report = report.with_name(f".previous-{STUDY}")
summary = pd.DataFrame()
if not (report / "settings.json").is_file() or not (report / "summary.csv").is_file():
    print("Нет сохранённого отчёта. Запусти соответствующий сценарий из папки проекта.")
else:
    metadata = read_json(report / "settings.json")
    summary = pd.read_csv(report / "summary.csv")
    print(f"{report} · COMSOL {metadata['comsol_run']}")
    display(summary)
    for name in ("pairs.csv", "spatial_diagnostics.csv"):
        path = report / name
        if path.is_file() and path.read_text(encoding="utf-8-sig").strip():
            display(pd.read_csv(path))
    figures = {
        "fit-effect": ("k_by_width.png", "k_by_fit.png"),
        "fit-effect-w04-w07": ("k_by_width.png", "k_by_fit.png"),
        "unclassified": ("k_by_width.png", "figures/jsi_overview.png"),
        "spatial-single": ("schmidt_number_vs_width.png", "figures/marginals_omega.png"),
        "pulse-series": ("K_overview.png", "K_vs_duration.png"),
    }
    for name in figures.get(STUDY, ()):
        path = report / name
        if path.is_file():
            display(Image(filename=str(path)))


In [ ]:
CASE_ROW = 0  # Номер строки сводки выше.

if not summary.empty and "npz" in summary:
    row = summary.iloc[CASE_ROW]
    with np.load(report / row["npz"], allow_pickle=False) as case:
        x = case["preview_omega_signal_rad_per_fs"]
        y = case["preview_omega_idler_rad_per_fs"]
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
        for ax, data, title, cmap in (
            (axes[0], case["plot_jsi"], "JSI на сетке превью", "inferno"),
            (axes[1], np.angle(case["jsa_preview"]), "Фаза JSA, рад", "twilight"),
        ):
            mesh = ax.pcolormesh(x, y, data, cmap=cmap, shading="auto")
            ax.set(title=title, xlabel="Signal, рад/фс", ylabel="Idler, рад/фс")
            fig.colorbar(mesh, ax=ax)
        method = row.get("method") or f"poly{row['degree']:g}"
        pump = row.get("pump_label", row["pump_mode"])
        fig.suptitle(f"{row['width_um']:g} мкм · PUMP {pump} · {method} · K={row['K']:.4g}")
        plt.show()